# Step 3: Download Images from TNG-100

This notebook downloads dark matter visualization images from the TNG-100 simulation for the selected subhalos.

## Process

1. **Setup Environment**: Configure Colab if needed
2. **Load Selected Subhalos**: Read the filtered catalog from Step 2
3. **Download Images**: Download cutout images in multiple resolutions
4. **Organize Data**: Structure images by resolution (224x224, 512x512)
5. **Generate Metadata**: Create tracking CSV with image paths

In [ ]:
# SETUP CELDA 1: Configure Google Colab environment
import os
import sys
from pathlib import Path

# Detect if running in Google Colab
try:
    import google.colab
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    print("🔧 Setting up Google Colab environment...\n")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    print("✓ Google Drive mounted at /content/drive")
    
    # Change to project directory
    project_path = '/content/drive/MyDrive/xai-dark-matter-localization'
    os.chdir(project_path)
    print(f"✓ Working directory: {os.getcwd()}")
    
    # Add to Python path
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print("✓ Python path configured\n")
else:
    print("⚠️  Not running in Colab. Using local environment.\n")

In [ ]:
# CELDA 7: Display download summary
print("\n" + "="*60)
print("DOWNLOAD SUMMARY")
print("="*60)

for resolution in IMAGE_RESOLUTIONS:
    success = download_results[resolution]["success"]
    failed = download_results[resolution]["failed"]
    total = success + failed
    success_rate = 100 * success / total if total > 0 else 0
    
    print(f"\n{resolution}x{resolution} resolution:")
    print(f"  Successful: {success}/{total} ({success_rate:.1f}%)")
    if failed > 0:
        print(f"  Failed: {failed}")

print(f"\n{'='*60}")
print(f"Dataset Statistics:")
print(f"  Total subhalos processed: {len(df_selected)}")
print(f"  Images with metadata: {len(df_metadata)}")
print(f"  Unique subhalos with images: {df_metadata['subhalo_id'].nunique()}")

# Count images per resolution
for res in IMAGE_RESOLUTIONS:
    count = len(df_metadata[df_metadata['resolution'] == res])
    print(f"  {res}x{res} images: {count}")

print(f"\nMetadata file: {metadata_file}")
print(f"\nNext step: Preprocess images (04_preprocess_images.ipynb)")
print("="*60)

## 6. Summary and Statistics

Display download results and dataset statistics.

In [ ]:
# CELDA 6: Generate metadata with image paths
metadata_rows = []

for _, row in tqdm(df_selected.iterrows(), desc="Creating metadata", total=len(df_selected)):
    subhalo_id = int(row["subhalo_id"])
    
    for resolution in IMAGE_RESOLUTIONS:
        image_path = image_dirs[resolution] / f"subhalo_{subhalo_id}.png"
        
        # Only include if image actually exists
        if image_path.exists():
            metadata_row = dict(row)
            metadata_row["resolution"] = resolution
            metadata_row["image_path"] = str(image_path.relative_to(DATA_ROOT.parent))
            metadata_rows.append(metadata_row)

# Create metadata dataframe
df_metadata = pd.DataFrame(metadata_rows)

# Save metadata
metadata_file = DATA_ROOT / "processed" / "TNG-DM-XAI-v1" / "metadata_with_images.csv"
metadata_file.parent.mkdir(parents=True, exist_ok=True)
df_metadata.to_csv(metadata_file, index=False)

print(f"✓ Created metadata with {len(df_metadata)} entries")
print(f"  Saved to: {metadata_file}")
print(f"\nMetadata structure:")
print(df_metadata.head())

## 5. Generate Metadata with Image Paths

Create a CSV with image file paths for each subhalo and resolution.

In [ ]:
# CELDA 5: Download images for each subhalo
download_results = {res: {"success": 0, "failed": 0} for res in IMAGE_RESOLUTIONS}

print(f"Starting image downloads...\n")

for idx, (_, row) in enumerate(tqdm(df_selected.iterrows(), total=len(df_selected), desc="Subhalos")):
    subhalo_id = int(row["subhalo_id"])
    subhalo_url = row["subhalo_url"]
    
    # Download each resolution
    for resolution in IMAGE_RESOLUTIONS:
        try:
            # Construct image URL
            image_url = subhalo_url + IMAGE_TYPES[resolution]
            
            # Create filename
            filename = f"subhalo_{subhalo_id}.png"
            output_path = image_dirs[resolution] / filename
            
            # Skip if already exists
            if output_path.exists():
                download_results[resolution]["success"] += 1
                continue
            
            # Download file
            download_file(image_url, output_path)
            download_results[resolution]["success"] += 1
            
        except Exception as e:
            download_results[resolution]["failed"] += 1
            if idx < 3:  # Only print first 3 errors
                print(f"  Error downloading {resolution}x{resolution} for subhalo {subhalo_id}: {str(e)[:50]}")

print("\n✓ Download phase complete")

## 4. Download Images from TNG API

⚠️ **Warning**: This step may take 1-2 hours depending on number of subhalos and API rate limits.
The API has rate limiting, so some requests may timeout (503 errors), but the process will continue.

In [ ]:
# CELDA 4: Configure image resolutions and directories
IMAGE_RESOLUTIONS = [224, 512]  # Resolutions to download
IMAGE_TYPES = {
    224: "cutouts/subhalo_224.png",
    512: "cutouts/subhalo_512.png"
}

# Create output directories for each resolution
image_dirs = {}
for res in IMAGE_RESOLUTIONS:
    image_dir = DATA_ROOT / "processed" / "TNG-DM-XAI-v1" / f"images_{res}"
    image_dir.mkdir(parents=True, exist_ok=True)
    image_dirs[res] = image_dir
    print(f"✓ Output directory for {res}x{res}: {image_dir}")

print(f"\nDownload configuration:")
print(f"  Resolutions: {IMAGE_RESOLUTIONS}")
print(f"  Total subhalos to download: {len(df_selected)}")
print(f"  Total images to download: {len(df_selected) * len(IMAGE_RESOLUTIONS)}")

## 3. Configure Download Parameters

Set image resolutions and output directories.

In [ ]:
# CELDA 3: Load the selected subhalos from step 2
selected_file = DATA_ROOT / "raw" / "tng" / "metadata_raw" / "subhalos_selected.csv"

if not selected_file.exists():
    raise FileNotFoundError(f"Selected catalog not found: {selected_file}")

df_selected = pd.read_csv(selected_file)
print(f"✓ Loaded {len(df_selected)} selected subhalos")
print(f"\nColumns: {list(df_selected.columns)}")
print(f"\nFirst few rows:")
print(df_selected[['subhalo_id', 'mass_log_msun', 'stellar_mass_api', 'sfr']].head())

## 2. Load Selected Subhalos

Read the catalog of selected subhalos from the previous step.

In [ ]:
# CELDA 2: Import required libraries and project modules
import pandas as pd
import numpy as np
from tqdm import tqdm
import sys
import importlib

# Force reload of modules (important in Colab)
for module in list(sys.modules.keys()):
    if 'src' in module:
        del sys.modules[module]

# Import project modules
from src.config import DATA_ROOT, DATASET_DIR, TNG_API_KEY
from src import tng_api
importlib.reload(tng_api)
from src.tng_api import download_file

print("✓ All libraries and modules imported successfully")
print(f"\nProject Configuration:")
print(f"  Data root: {DATA_ROOT}")
print(f"  Dataset directory: {DATASET_DIR}")
print(f"  API key configured: {'✓' if TNG_API_KEY else '✗'}")

## 1. Import Libraries and Modules